In [6]:
# Cellule 0 — Installation des dépendances (à exécuter une seule fois)
import sys
!{sys.executable} -m pip install scikit-learn imbalanced-learn --quiet
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Shift 1 --- Brice 
df = pd.read_csv("star_classification.csv")

# Shape et types
print(df.shape)
print(df.dtypes)
print(df.head())
# Valeurs manquantes
print("Les valeurs manquantes:", df.isnull().sum(), " #fin des valeurs manquantes")


# Distribution des classes
print("Distribution des classes:")
print(df['class'].value_counts())
df['class'].value_counts().plot(kind='bar')
plt.title("Distribution des classes")
plt.show()

# Statistiques descriptives
print("La description statistique des données:")
print(df.describe())

# Distribution du redshift par classe
sns.boxplot(x='class', y='redshift', data=df)
plt.title("Redshift par classe")
plt.show()

# La supression des colonnes inutiles
colonnes_a_supprimer = ['obj_ID', 'rerun_ID', 'run_ID', 'cam_col', 
                    'field_ID', 'spec_obj_ID', 'plate', 'MJD', 'fiber_ID']
print("Colonnes qui ne sont pas utiles pour la classification:", colonnes_a_supprimer)

df = df.drop(columns=colonnes_a_supprimer)

print("Colonnes restantes:", df.columns.tolist())
print("Shape après nettoyage:", df.shape)

print(df.head())


(100000, 18)
obj_ID         float64
alpha          float64
delta          float64
u              float64
g              float64
r              float64
i              float64
z              float64
run_ID           int64
rerun_ID         int64
cam_col          int64
field_ID         int64
spec_obj_ID    float64
class              str
redshift       float64
plate            int64
MJD              int64
fiber_ID         int64
dtype: object
         obj_ID       alpha      delta         u         g         r  \
0  1.237661e+18  135.689107  32.494632  23.87882  22.27530  20.39501   
1  1.237665e+18  144.826101  31.274185  24.77759  22.83188  22.58444   
2  1.237661e+18  142.188790  35.582444  25.26307  22.66389  20.60976   
3  1.237663e+18  338.741038  -0.402828  22.13682  23.77656  21.61162   
4  1.237680e+18  345.282593  21.183866  19.43718  17.58028  16.49747   

          i         z  run_ID  rerun_ID  cam_col  field_ID   spec_obj_ID  \
0  19.16573  18.79371    3606       301        2  

/tmp/ipykernel_25927/3439007629.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Colonnes qui ne sont pas utiles pour la classification: ['obj_ID', 'rerun_ID', 'run_ID', 'cam_col', 'field_ID', 'spec_obj_ID', 'plate', 'MJD', 'fiber_ID']
Colonnes restantes: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'class', 'redshift']
Shape après nettoyage: (100000, 9)
        alpha      delta         u         g         r         i         z  \
0  135.689107  32.494632  23.87882  22.27530  20.39501  19.16573  18.79371   
1  144.826101  31.274185  24.77759  22.83188  22.58444  21.16812  21.61427   
2  142.188790  35.582444  25.26307  22.66389  20.60976  19.34857  18.94827   
3  338.741038  -0.402828  22.13682  23.77656  21.61162  20.50454  19.25010   
4  345.282593  21.183866  19.43718  17.58028  16.49747  15.97711  15.54461   

    class  redshift  
0  GALAXY  0.634794  
1  GALAXY  0.779136  
2  GALAXY  0.644195  
3  GALAXY  0.932346  
4  GALAXY  0.116123  


/tmp/ipykernel_25927/3439007629.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:

# Shift 1 --- Sarah T
# ============================================================
# Feature Engineering & Préparation finale du dataset
# ============================================================

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE

# --- 1. Création des indices de couleur (déjà amorcée) ---
df['u-g'] = df['u'] - df['g']
df['g-r'] = df['g'] - df['r']
df['r-i'] = df['r'] - df['i']
df['i-z'] = df['i'] - df['z']

print("Shape après feature engineering:", df.shape)
print("Nouvelles colonnes:", ['u-g', 'g-r', 'r-i', 'i-z'])
print(df[['u-g', 'g-r', 'r-i', 'i-z']].describe())

# --- 2. Visualisation des indices de couleur par classe ---
color_features = ['u-g', 'g-r', 'r-i', 'i-z']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(color_features):
    sns.boxplot(x='class', y=feat, data=df, ax=axes[i],
                order=['GALAXY', 'STAR', 'QSO'],
                palette={'GALAXY': '#4C72B0', 'STAR': '#DD8452', 'QSO': '#55A868'})
    axes[i].set_title(f"Indice de couleur : {feat}", fontsize=13)
    axes[i].set_xlabel("Classe")
    axes[i].set_ylabel(feat)

plt.suptitle("Distribution des indices de couleur par classe", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("color_indices_boxplots.png", dpi=150)
plt.show()
print("→ Boxplots des indices de couleur enregistrés.")

# --- 3. Pairplot pour visualiser la séparabilité ---
sample = df.sample(2000, random_state=42)  # sous-échantillon pour la lisibilité

sns.pairplot(
    sample[color_features + ['class']],
    hue='class',
    hue_order=['GALAXY', 'STAR', 'QSO'],
    palette={'GALAXY': '#4C72B0', 'STAR': '#DD8452', 'QSO': '#55A868'},
    plot_kws={'alpha': 0.4, 's': 15},
    diag_kind='kde'
)
plt.suptitle("Pairplot des indices de couleur", y=1.02, fontsize=14, fontweight='bold')
plt.savefig("color_indices_pairplot.png", dpi=120)
plt.show()
print("→ Pairplot enregistré.")

# --- 4. Heatmap de corrélation (features complètes) ---
features_for_corr = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z',
                     'redshift', 'u-g', 'g-r', 'r-i', 'i-z']

plt.figure(figsize=(12, 9))
corr = df[features_for_corr].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={"size": 9})
plt.title("Matrice de corrélation — Features physiques + indices de couleur",
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.show()
print("→ Heatmap de corrélation enregistrée.")

# --- 5. Encodage de la variable cible ---
le = LabelEncoder()
df['class_encoded'] = le.fit_transform(df['class'])
print("\nEncodage des classes:")
for cls, code in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {cls} → {code}")

# --- 6. Séparation features / cible ---
feature_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z',
                'redshift', 'u-g', 'g-r', 'r-i', 'i-z']

X = df[feature_cols]
y = df['class_encoded']

print(f"\nX shape : {X.shape}")
print(f"y distribution avant SMOTE :\n{pd.Series(y).value_counts()}")

# --- 7. Split stratifié 80/20 ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"\nTrain : {X_train.shape[0]} observations")
print(f"Test  : {X_test.shape[0]} observations")

# --- 8. Normalisation (StandardScaler sur le train, appliqué sur le test) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\nNormalisation appliquée (fit sur train uniquement).")
print(f"Moyenne train (doit être ≈0) : {X_train_scaled.mean(axis=0).round(4)}")
print(f"Écart-type train (doit être ≈1): {X_train_scaled.std(axis=0).round(4)}")

# --- 9. Application de SMOTE (sur le train uniquement) ---
print(f"\nDistribution avant SMOTE :\n{pd.Series(y_train).value_counts()}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Distribution après SMOTE  :\n{pd.Series(y_train_resampled).value_counts()}")
print(f"Shape final X_train : {X_train_resampled.shape}")

# --- 10. Visualisation SMOTE avant/après ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

class_names = le.classes_
counts_before = pd.Series(y_train).value_counts().sort_index()
counts_after  = pd.Series(y_train_resampled).value_counts().sort_index()
colors = ['#4C72B0', '#55A868', '#DD8452']

axes[0].bar(class_names, counts_before, color=colors)
axes[0].set_title("Avant SMOTE (train)", fontsize=12)
axes[0].set_ylabel("Nombre d'observations")
for j, v in enumerate(counts_before):
    axes[0].text(j, v + 200, str(v), ha='center', fontweight='bold')

axes[1].bar(class_names, counts_after, color=colors)
axes[1].set_title("Après SMOTE (train)", fontsize=12)
axes[1].set_ylabel("Nombre d'observations")
for j, v in enumerate(counts_after):
    axes[1].text(j, v + 200, str(v), ha='center', fontweight='bold')

plt.suptitle("Rééquilibrage des classes avec SMOTE", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("smote_distribution.png", dpi=150)
plt.show()
print("→ Graphique SMOTE enregistré.")

# --- 11. Récapitulatif final ---
print("\n" + "="*55)
print("✅  SPRINT 1 — Données prêtes pour la modélisation")
print("="*55)
print(f"  Features utilisées    : {feature_cols}")
print(f"  X_train (post-SMOTE)  : {X_train_resampled.shape}")
print(f"  X_test  (non rééch.)  : {X_test_scaled.shape}")
print(f"  Classes encodées      : {dict(zip(le.classes_, le.transform(le.classes_)))}")
print("="*55)


Shape après feature engineering: (100000, 14)
Nouvelles colonnes: ['u-g', 'g-r', 'r-i', 'i-z']
                 u-g            g-r            r-i            i-z
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        1.449081       0.885625       0.560908       0.416044
std         1.179335      31.688694       0.501517      31.678379
min       -12.748140  -10017.165600     -14.649070     -13.162490
25%         0.631215       0.379798       0.216317       0.134500
50%         1.321670       0.931450       0.479795       0.338040
75%         2.044558       1.577983       0.892192       0.474080
max        18.624950      14.315170      12.205800   10017.016750


/tmp/ipykernel_25927/1236875488.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='class', y=feat, data=df, ax=axes[i],
/tmp/ipykernel_25927/1236875488.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='class', y=feat, data=df, ax=axes[i],
/tmp/ipykernel_25927/1236875488.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='class', y=feat, data=df, ax=axes[i],
/tmp/ipykernel_25927/1236875488.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `

→ Boxplots des indices de couleur enregistrés.


/tmp/ipykernel_25927/1236875488.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


→ Pairplot enregistré.


/tmp/ipykernel_25927/1236875488.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


→ Heatmap de corrélation enregistrée.

Encodage des classes:
  GALAXY → 0
  QSO → 1
  STAR → 2

X shape : (100000, 12)
y distribution avant SMOTE :
class_encoded
0    59445
2    21594
1    18961
Name: count, dtype: int64

Train : 80000 observations
Test  : 20000 observations

Normalisation appliquée (fit sur train uniquement).
Moyenne train (doit être ≈0) : [ 0. -0. -0.  0. -0. -0.  0. -0.  0.  0. -0. -0.]
Écart-type train (doit être ≈1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Distribution avant SMOTE :
class_encoded
0    47556
2    17275
1    15169
Name: count, dtype: int64
Distribution après SMOTE  :
class_encoded
1    47556
2    47556
0    47556
Name: count, dtype: int64
Shape final X_train : (142668, 12)
→ Graphique SMOTE enregistré.

✅  SPRINT 1 — Données prêtes pour la modélisation
  Features utilisées    : ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'u-g', 'g-r', 'r-i', 'i-z']
  X_train (post-SMOTE)  : (142668, 12)
  X_test  (non rééch.)  : (20000, 12)
  Classes enco

/tmp/ipykernel_25927/1236875488.py:141: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# ============================================================
# Sprint 2 — Tantely Sarah
# Modèle XGBoost — Classification d'objets célestes
# ============================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import time

from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import cross_val_score

# ── Les variables ci-dessous viennent du Sprint 1 ──────────
# X_train_resampled, y_train_resampled  (post-SMOTE, normalisé)
# X_test_scaled, y_test                (non rééquilibré, normalisé)
# le  (LabelEncoder)  →  le.classes_ = ['GALAXY', 'QSO', 'STAR']
# ────────────────────────────────────────────────────────────

CLASS_NAMES = list(le.classes_)   # ['GALAXY', 'QSO', 'STAR']
PALETTE     = {'GALAXY': '#4C72B0', 'QSO': '#55A868', 'STAR': '#DD8452'}
COLORS      = [PALETTE[c] for c in CLASS_NAMES]


# ── 1. ENTRAÎNEMENT ─────────────────────────────────────────
print("=" * 55)
print("  SPRINT 2 — XGBoost")
print("=" * 55)

xgb = XGBClassifier(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.1,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    use_label_encoder = False,
    eval_metric       = 'mlogloss',
    random_state      = 42,
    n_jobs            = -1
)

print("\n[1/5] Entraînement XGBoost...")
t0 = time.time()
xgb.fit(X_train_resampled, y_train_resampled)
train_time = time.time() - t0
print(f"      Terminé en {train_time:.1f}s")


# ── 2. PRÉDICTIONS ──────────────────────────────────────────
print("\n[2/5] Prédictions sur le jeu de test...")
y_pred      = xgb.predict(X_test_scaled)
y_pred_prob = xgb.predict_proba(X_test_scaled)   # shape (n, 3)


# ── 3. MÉTRIQUES ────────────────────────────────────────────
print("\n[3/5] Calcul des métriques...")

f1_macro  = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')
f1_per_class = f1_score(y_test, y_pred, average=None)

# AUC-ROC one-vs-rest
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
auc_roc    = roc_auc_score(y_test_bin, y_pred_prob,
                            multi_class='ovr', average='macro')

print(f"\n  F1 macro    : {f1_macro:.4f}")
print(f"  F1 weighted : {f1_weighted:.4f}")
print(f"  AUC-ROC OvR : {auc_roc:.4f}")
print(f"  Temps train : {train_time:.1f}s")

print("\n  F1 par classe :")
for name, score in zip(CLASS_NAMES, f1_per_class):
    print(f"    {name:<8} : {score:.4f}")

print("\n  Rapport complet :")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

# Dictionnaire de résultats (pour le tableau comparatif Sprint 4)
xgb_results = {
    'model'       : 'XGBoost',
    'f1_macro'    : round(f1_macro, 4),
    'f1_weighted' : round(f1_weighted, 4),
    'auc_roc'     : round(auc_roc, 4),
    'train_time'  : round(train_time, 1),
    'f1_per_class': {n: round(s, 4) for n, s in zip(CLASS_NAMES, f1_per_class)}
}


# ── 4. VALIDATION CROISÉE (5 plis) ──────────────────────────
print("\n[4/5] Validation croisée 5 plis (sur données SMOTE)...")
cv_scores = cross_val_score(
    xgb, X_train_resampled, y_train_resampled,
    cv=5, scoring='f1_macro', n_jobs=-1
)
print(f"  CV F1 macro : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Scores      : {[round(s,4) for s in cv_scores]}")
xgb_results['cv_f1_mean'] = round(cv_scores.mean(), 4)
xgb_results['cv_f1_std']  = round(cv_scores.std(),  4)


# ── 5. VISUALISATIONS ───────────────────────────────────────
print("\n[5/5] Génération des visualisations...")
import os
os.makedirs("outputs", exist_ok=True)

# -- 5a. Matrice de confusion --
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # normalisée en ligne

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2%'],
    ['Matrice de confusion — Valeurs absolues',
     'Matrice de confusion — Normalisée (recall par classe)']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, ax=ax, cbar=True)
    ax.set_xlabel("Prédit",   fontsize=12)
    ax.set_ylabel("Réel",     fontsize=12)
    ax.set_title(title,       fontsize=12, fontweight='bold')

plt.suptitle("XGBoost — Matrices de confusion", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/xgb_confusion_matrix.png", dpi=150)
plt.close()
print("  → confusion_matrix sauvegardée")

# -- 5b. F1 par classe (barplot) --
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(CLASS_NAMES, f1_per_class, color=COLORS, edgecolor='white', width=0.5)
ax.axhline(f1_macro, color='gray', linestyle='--', linewidth=1.2, label=f'F1 macro = {f1_macro:.3f}')
for bar, score in zip(bars, f1_per_class):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{score:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score F1")
ax.set_title("XGBoost — F1 par classe", fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("outputs/xgb_f1_per_class.png", dpi=150)
plt.close()
print("  → f1_per_class sauvegardée")

# -- 5c. Courbes ROC one-vs-rest --
fig, ax = plt.subplots(figsize=(7, 6))
for i, (name, color) in enumerate(zip(CLASS_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    auc_i = roc_auc_score(y_test_bin[:, i], y_pred_prob[:, i])
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc_i:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aléatoire')
ax.set_xlabel("Taux de faux positifs (FPR)")
ax.set_ylabel("Taux de vrais positifs (TPR)")
ax.set_title("XGBoost — Courbes ROC (one-vs-rest)", fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.01])
plt.tight_layout()
plt.savefig("outputs/xgb_roc_curves.png", dpi=150)
plt.close()
print("  → roc_curves sauvegardée")

# -- 5d. Importance des variables --
feature_names = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z',
                 'redshift', 'u-g', 'g-r', 'r-i', 'i-z']
importances = xgb.feature_importances_
idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([feature_names[i] for i in idx], importances[idx],
       color='#4C72B0', edgecolor='white')
ax.set_ylabel("Importance (gain)")
ax.set_title("XGBoost — Importance des variables", fontsize=13, fontweight='bold')
ax.set_xticklabels([feature_names[i] for i in idx], rotation=30, ha='right')
plt.tight_layout()
plt.savefig("outputs/xgb_feature_importance.png", dpi=150)
plt.close()
print("  → feature_importance sauvegardée")

# ── RÉSUMÉ FINAL ────────────────────────────────────────────
print("\n" + "=" * 55)
print("  XGBoost — Résultats finaux")
print("=" * 55)
for k, v in xgb_results.items():
    print(f"  {k:<15} : {v}")
print("=" * 55)
print("\n✅  Sprint 2 terminé. xgb_results prêt pour Sprint 4.")

  SPRINT 2 — XGBoost

[1/5] Entraînement XGBoost...


/home/hp/Documents/stellar-classification/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:43:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


      Terminé en 16.7s

[2/5] Prédictions sur le jeu de test...

[3/5] Calcul des métriques...

  F1 macro    : 0.9745
  F1 weighted : 0.9777
  AUC-ROC OvR : 0.9964
  Temps train : 16.7s

  F1 par classe :
    GALAXY   : 0.9812
    QSO      : 0.9493
    STAR     : 0.9930

  Rapport complet :
              precision    recall  f1-score   support

      GALAXY       0.98      0.98      0.98     11889
         QSO       0.95      0.95      0.95      3792
        STAR       0.99      1.00      0.99      4319

    accuracy                           0.98     20000
   macro avg       0.97      0.98      0.97     20000
weighted avg       0.98      0.98      0.98     20000


[4/5] Validation croisée 5 plis (sur données SMOTE)...


/home/hp/Documents/stellar-classification/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:43:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/hp/Documents/stellar-classification/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:43:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/hp/Documents/stellar-classification/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:43:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/hp/Documents/stellar-classification/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:43:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Paramet

  CV F1 macro : 0.9804 ± 0.0010
  Scores      : [np.float64(0.9789), np.float64(0.9796), np.float64(0.9808), np.float64(0.981), np.float64(0.9816)]

[5/5] Génération des visualisations...
  → confusion_matrix sauvegardée
  → f1_per_class sauvegardée
  → roc_curves sauvegardée


/tmp/ipykernel_25927/4227359523.py:187: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([feature_names[i] for i in idx], rotation=30, ha='right')


  → feature_importance sauvegardée

  XGBoost — Résultats finaux
  model           : XGBoost
  f1_macro        : 0.9745
  f1_weighted     : 0.9777
  auc_roc         : 0.9964
  train_time      : 16.7
  f1_per_class    : {'GALAXY': np.float64(0.9812), 'QSO': np.float64(0.9493), 'STAR': np.float64(0.993)}
  cv_f1_mean      : 0.9804
  cv_f1_std       : 0.001

✅  Sprint 2 terminé. xgb_results prêt pour Sprint 4.
